<a href="https://colab.research.google.com/github/esraozkan98-commits/soccer-analysis/blob/main/soccer_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install db-dtypes -q
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
PROJECT = "primordial-mile-449805-j0"
client = bigquery.Client(project=PROJECT)

def sorgu(sql):
    return client.query(sql).to_dataframe(create_bqstorage_client=False)

df = sorgu("SELECT * FROM `primordial-mile-449805-j0.dbt_soccer.match_analysis`")
print(df.shape)
df.head()

(25979, 25)


,league_name,season,home_team_api_id,away_team_api_id,home_team_goal,away_team_goal,result,total_goals,goal_difference,B365H,...,away_rating,rating_diff,home_possession,away_possession,home_cards,away_cards,home_fouls,away_fouls,home_shots,away_shots
0,France Ligue 1,2015/2016,10242,9847,0,9,away_win,9,-9,12.0,...,78.082476,-12.579578,41.0,59.0,2.0,0.0,9.0,7.0,8.0,7.0
1,Germany 1. Bundesliga,2010/2011,8152,9823,1,8,away_win,9,-7,11.0,...,81.368429,-13.812860,49.0,51.0,3.0,2.0,15.0,14.0,5.0,5.0
2,Spain LIGA BBVA,2014/2015,9783,8633,2,8,away_win,10,-6,9.5,...,83.345889,-11.340837,41.0,59.0,1.0,1.0,8.0,9.0,5.0,7.0
3,France Ligue 1,2014/2015,9747,9831,2,7,away_win,9,-5,2.2,...,68.453597,0.028570,46.0,54.0,0.0,1.0,13.0,14.0,7.0,5.0
4,England Premier League,2013/2014,8344,8650,3,6,away_win,9,-3,9.5,...,77.925863,-6.160717,33.0,67.0,2.0,3.0,14.0,11.0,3.0,6.0


Hipotez 1 — Ev Sahibi Avantajı
**Soru:** Ev sahibi olmak kazanma şansını gerçekten artırıyor mu, yoksa fark tesadüf mü?
**H0:** Ev/beraberlik/deplasman sonuçları eşit dağılır (avantaj yok).
**H1:** Dağılım eşit değildir; ev sahibi anlamlı şekilde fazla kazanır.
**Test:** Ki-kare uyum iyiliği testi (chi-square goodness of fit).


In [ ]:
from scipy import stats

sayim = df['result'].value_counts()
n = sayim.sum()
print("Sonuç sayıları:\n", sayim.to_string())
print("\nOranlar:\n", (sayim/n).round(3).to_string())

# H0: avantaj yoksa üç sonuç da eşit (n/3) beklenir
gozlenen = [sayim['home_win'], sayim['draw'], sayim['away_win']]
beklenen = [n/3, n/3, n/3]
chi2, p = stats.chisquare(gozlenen, beklenen)

print(f"\nKi-kare = {chi2:.1f}")
print(f"p-değeri = {p:.2e}")
print("Karar:", "H0 RED → ev avantajı İSTATİSTİKSEL OLARAK ANLAMLI" if p < 0.05 else "H0 reddedilemez")

Sonuç sayıları:
 result
home_win    11917
away_win     7466
draw         6596

Oranlar:
 result
home_win    0.459
away_win    0.287
draw        0.254

Ki-kare = 1881.6
p-değeri = 0.00e+00
Karar: H0 RED → ev avantajı İSTATİSTİKSEL OLARAK ANLAMLI


In [ ]:
import pandas as pd
import plotly.express as px

vc = df['result'].value_counts(normalize=True)
d = pd.DataFrame({
    'Sonuç': vc.index.map({'home_win':'Ev Sahibi','draw':'Beraberlik','away_win':'Deplasman'}),
    'Oran': vc.values
})
fig = px.bar(d, x='Sonuç', y='Oran',
             text=d['Oran'].apply(lambda v: f"{v:.1%}"),
             color='Sonuç',
             color_discrete_map={'Ev Sahibi':'#2E7D32','Beraberlik':'#B0BEC5','Deplasman':'#5C8AA8'},
             title='Maç Sonuçlarının Dağılımı (2008–2016)')
fig.update_layout(showlegend=False, yaxis_tickformat='.0%', plot_bgcolor='white')
fig.show()

### Hipotez 2 — Bahis Favorisi
**Soru:** Bahis favorisi (en düşük oranlı sonuç) gerçekten kazanıyor mu, yoksa basit tahminden iyi değil mi?
**H0:** Favori, "hep ev sahibini seç" stratejisinden daha iyi tutturmaz.
**H1:** Favori, bu basit temeli anlamlı şekilde geçer.
**Test:** Binom testi (tek yönlü).

In [ ]:
from scipy import stats

bet = df.dropna(subset=['favorite_correct'])     # bahis oranı olan maçlar
n = len(bet)
isabet = int(bet['favorite_correct'].sum())
oran = isabet / n

hep_ev = (df['result'] == 'home_win').mean()      # basit temel: hep ev sahibi
print(f"Bahis verili maç sayısı: {n}")
print(f"Favori tutturma oranı: {oran:.1%}")
print(f"'Hep ev sahibi' temeli: {hep_ev:.1%}")

# Favori, hep-ev temelini geçiyor mu?
test = stats.binomtest(isabet, n, p=hep_ev, alternative='greater')
print(f"\np-değeri = {test.pvalue:.2e}")
print("Karar:", "H0 RED → favori basit tahmini ANLAMLI şekilde geçiyor"
      if test.pvalue < 0.05 else "H0 reddedilemez")

Bahis verili maç sayısı: 22592
Favori tutturma oranı: 53.3%
'Hep ev sahibi' temeli: 45.9%

p-değeri = 8.76e-112
Karar: H0 RED → favori basit tahmini ANLAMLI şekilde geçiyor


In [ ]:
import pandas as pd
import plotly.express as px

karsilastirma = pd.DataFrame({
    'Strateji': ['Rastgele (1/3)', 'Hep Ev Sahibi', 'Bahis Favorisi'],
    'İsabet': [1/3, hep_ev, oran]
})
fig = px.bar(karsilastirma, x='Strateji', y='İsabet',
             text=karsilastirma['İsabet'].apply(lambda v: f"{v:.1%}"),
             color='Strateji',
             color_discrete_map={'Rastgele (1/3)':'#B0BEC5',
                                 'Hep Ev Sahibi':'#5C8AA8',
                                 'Bahis Favorisi':'#2E7D32'},
             title='Tahmin Stratejilerinin İsabet Oranı')
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, yaxis_tickformat='.0%', plot_bgcolor='white')
fig.show()

### Hipotez 3 — Ev Sahibi Olmak Oyun Tarzını Etkiliyor mu?
**Soru:** Ev sahibi takımlar topa daha çok mu sahip oluyor, daha az mı faul/kart görüyor?
**H0:** Ev ve deplasman değerleri (possession/faul/kart) eşittir.
**H1:** Aralarında anlamlı fark vardır.
**Test:** Eşleştirilmiş t-testi (paired t-test), çünkü değerler aynı maçtan çift olarak geliyor.

In [ ]:
from scipy import stats

def esli_test(ev_kol, dep_kol, ad):
    alt = df.dropna(subset=[ev_kol, dep_kol])   # ikisi de dolu olan maçlar
    ev, dep = alt[ev_kol].mean(), alt[dep_kol].mean()
    t, p = stats.ttest_rel(alt[ev_kol], alt[dep_kol])   # eşleştirilmiş test
    print(f"{ad}: Ev={ev:.2f}  Deplasman={dep:.2f}  fark={ev-dep:+.2f}")
    print(f"   t={t:.1f}, p={p:.2e} →", "ANLAMLI" if p < 0.05 else "anlamsız", f"(n={len(alt)})\n")

esli_test('home_possession', 'away_possession', 'Topa sahip olma (%)')
esli_test('home_fouls',      'away_fouls',      'Faul sayısı')
esli_test('home_cards',      'away_cards',      'Kart sayısı')

Topa sahip olma (%): Ev=51.50  Deplasman=48.50  fark=+2.99
   t=14.8, p=1.02e-48 → ANLAMLI (n=8419)

Faul sayısı: Ev=7.53  Deplasman=7.86  fark=-0.33
   t=-9.7, p=4.44e-22 → ANLAMLI (n=14217)

Kart sayısı: Ev=1.99  Deplasman=2.36  fark=-0.37
   t=-24.7, p=1.46e-131 → ANLAMLI (n=14217)



In [ ]:
import pandas as pd
import plotly.express as px

alt = df.dropna(subset=['home_possession','away_possession'])
poss = pd.DataFrame({'Taraf':['Ev Sahibi','Deplasman'],
                     'Possession':[alt.home_possession.mean(), alt.away_possession.mean()]})
fig = px.bar(poss, x='Taraf', y='Possession',
             text=poss['Possession'].apply(lambda v: f"{v:.1f}%"),
             color='Taraf', color_discrete_map={'Ev Sahibi':'#2E7D32','Deplasman':'#5C8AA8'},
             title='Ortalama Topa Sahip Olma (%)')
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, plot_bgcolor='white')
fig.show()

### Hipotez 4 — Takım Kalitesi Ev Avantajını Değiştiriyor mu?
**Soru:** Reyting farkı (ev - deplasman) sonucu ne kadar belirliyor? Denk takımlarda ev avantajı sürüyor mu?
**H0:** Reyting farkının ev galibiyetine etkisi yoktur.
**H1:** Reyting farkı arttıkça ev galibiyet olasılığı anlamlı şekilde artar; ama denk takımlarda bile ev avantajı kalır.
**Test:** Pearson korelasyonu + Lojistik regresyon.

In [ ]:
from scipy import stats
alt = df.dropna(subset=['rating_diff','goal_difference'])
r, p = stats.pearsonr(alt['rating_diff'], alt['goal_difference'])
print(f"Reyting farkı ↔ gol farkı korelasyonu: r = {r:.3f}, p = {p:.2e}")
print("→ Pozitif: güçlü takım daha farklı kazanıyor" if r > 0 else "")

Reyting farkı ↔ gol farkı korelasyonu: r = 0.451, p = 0.00e+00
→ Pozitif: güçlü takım daha farklı kazanıyor


In [ ]:
import pandas as pd
import plotly.express as px

alt = df.dropna(subset=['rating_diff']).copy()
alt['ev_galip'] = (alt['result'] == 'home_win').astype(int)

kesme  = [-100, -5, -2, 2, 5, 100]
etiket = ['Ev çok zayıf', 'Ev zayıf', 'Denk', 'Ev güçlü', 'Ev çok güçlü']
alt['grup'] = pd.cut(alt['rating_diff'], bins=kesme, labels=etiket)

oran = alt.groupby('grup', observed=True)['ev_galip'].mean().reset_index()
fig = px.bar(oran, x='grup', y='ev_galip',
             text=oran['ev_galip'].apply(lambda v: f"{v:.1%}"),
             color_discrete_sequence=['#2E7D32'],
             title='Rating Farkına Göre Ev Sahibi Galibiyet Oranı')
fig.update_traces(textposition='outside')
fig.update_layout(yaxis_tickformat='.0%', plot_bgcolor='white',
                  xaxis_title='Rating farkı (ev − deplasman)', yaxis_title='Ev galibiyet oranı')
fig.show()

In [ ]:
import statsmodels.formula.api as smf
import numpy as np

alt = df.dropna(subset=['rating_diff']).copy()
alt['ev_galip'] = (alt['result'] == 'home_win').astype(int)

model = smf.logit('ev_galip ~ rating_diff', data=alt).fit(disp=0)
print(model.summary())

# Denk takımlarda (rating farkı = 0) ev galibiyet olasılığı
b0 = model.params['Intercept']
denk = 1 / (1 + np.exp(-b0))
print(f"\n>> Denk takımlarda ev galibiyet olasılığı: {denk:.1%}")

                           Logit Regression Results                           
Dep. Variable:               ev_galip   No. Observations:                25221
Model:                          Logit   Df Residuals:                    25219
Method:                           MLE   Df Model:                            1
Date:                Mon, 15 Jun 2026   Pseudo R-squ.:                  0.1024
Time:                        14:33:45   Log-Likelihood:                -15612.
converged:                       True   LL-Null:                       -17393.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      -0.2067      0.014    -15.179      0.000      -0.233      -0.180
rating_diff     0.1806      0.003     53.778      0.000       0.174       0.187

>> Denk takımlarda ev galibiyet olasılığı: 44.9

## Makine Öğrenmesi — Maç Sonucu Tahmini

**Soru:** Bir maçın sonucunu (ev / beraberlik / deplasman) önceden tahmin edebilir miyiz? Kendi modelimiz, bahisçinin favorisini (%53,3) yakalayabilir mi?

**Özellikler (girdiler):** reyting farkı + bahis oranları
**Hedef:** maç sonucu
**Yöntem:** Veriyi %80 eğitim / %20 test diye böl → lojistik regresyon ve random forest eğit → doğruluğu baz çizgileriyle (rastgele %33, hep-ev %45,9, bahis favorisi %53,3) kıyasla.

In [ ]:
from sklearn.model_selection import train_test_split
veri = df.dropna(subset=['rating_diff','B365H','B365D','B365A']).copy()
X = veri[['rating_diff','B365H','B365D','B365A']]
y = veri['result']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Eğitim:", X_train.shape[0], "| Test:", X_test.shape[0])

Eğitim: 17661 | Test: 4416


### Lojistik Regresyon

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
m1 = LogisticRegression(max_iter=1000).fit(X_train[['rating_diff']], y_train)
acc1 = accuracy_score(y_test, m1.predict(X_test[['rating_diff']]))
m2 = LogisticRegression(max_iter=1000).fit(X_train, y_train)
acc2 = accuracy_score(y_test, m2.predict(X_test))
test = veri.loc[y_test.index]
print(f"Rastgele           : {1/3:.1%}")
print(f"Hep ev sahibi      : {(y_test=='home_win').mean():.1%}")
print(f"Bahis favorisi     : {test['favorite_correct'].mean():.1%}")
print(f"Lojistik (reyting) : {acc1:.1%}")
print(f"Lojistik (r+oran)  : {acc2:.1%}")

Rastgele           : 33.3%
Hep ev sahibi      : 45.9%
Bahis favorisi     : 52.9%
Lojistik (reyting) : 53.3%
Lojistik (r+oran)  : 53.5%


### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_train, y_train)
acc_rf = accuracy_score(y_test, rf.predict(X_test))
print(f"Random Forest: {acc_rf:.1%}")
print(pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).round(3).to_string())

Random Forest: 45.4%
rating_diff    0.585
B365A          0.150
B365H          0.149
B365D          0.117


### Yöntemlerin Karşılaştırması

In [ ]:
import plotly.express as px
sonuclar = pd.DataFrame({
    'Yöntem': ['Rastgele','Hep Ev','Bahis Favorisi','Lojistik (reyting)','Lojistik (r+oran)','Random Forest'],
    'Doğruluk': [1/3, (y_test=='home_win').mean(), test['favorite_correct'].mean(), acc1, acc2, acc_rf]
})
fig = px.bar(sonuclar, x='Yöntem', y='Doğruluk',
             text=sonuclar['Doğruluk'].apply(lambda v: f"{v:.1%}"),
             color_discrete_sequence=['#2E7D32'],
             title='Tahmin Yöntemlerinin Doğruluğu')
fig.update_traces(textposition='outside')
fig.update_layout(yaxis_tickformat='.0%', plot_bgcolor='white', xaxis_tickangle=-30)
fig.show()